In [ ]:
import json
import csv
import sys
from google.colab import files
import glob as gl

JSON_FILE = gl.glob('*.json')[0]

with open(JSON_FILE, 'r', encoding='utf-8') as f:
  data = json.load(f)

CSV_FILE = JSON_FILE.replace('.json', '.csv')

processed_accessions = set()

with open(CSV_FILE, 'w', newline='', encoding='utf-8-sig') as csvfile:
  headers = [
      'GISAID_Accession',
      'GISAID_Isolate',
      'GISAID_Collection_Date',
      'GISAID_Originating_Lab',
      'GISAID_Submitting_Lab',
      'GISAID_Authors'
  ]

  writer = csv.DictWriter(csvfile, fieldnames=headers)
  writer.writeheader()

  for item in data:
    content = item['content']
    lines = content.split('\n')
    accession = ''
    isolate = ''
    collection_date = ''
    originating_lab = ''
    submitting_lab = ''
    authors = ''

    for line in lines:
      line = line.strip()

      if line.startswith('EPI_ISL_') or line.startswith('EPI_SET_'):
        accession = line
      elif line.startswith('Virus name:'):
        isolate = line.replace('Virus name:', '').strip()
      elif line.startswith('Collection date:'):
        collection_date = line.replace('Collection date:', '').strip()
      elif line.startswith('Originating Lab:'):
        originating_lab = line.replace('Originating Lab:', '').strip()
      elif line.startswith('Submitting Lab:'):
        submitting_lab = line.replace('Submitting Lab:', '').strip()
      elif line.startswith('Authors:'):
        authors = line.replace('Authors:', '').strip()
      elif line.startswith('Authors'):
        authors = line.replace('Authors', '').strip()

    if not accession and 'callId' in item:
      accession = item['callId'].split('/')[0]

    if accession in processed_accessions:
      continue

    processed_accessions.add(accession)

    csv_row = {
        'GISAID_Accession': accession,
        'GISAID_Isolate': isolate,
        'GISAID_Collection_Date': collection_date,
        'GISAID_Originating_Lab': originating_lab,
        'GISAID_Submitting_Lab': submitting_lab,
        'GISAID_Authors': authors
    }

    writer.writerow(csv_row)

for i in gl.glob('*.csv'):
  files.download(i)